In [22]:
from pydantic import BaseModel
from typing import List

class student(BaseModel):
    name:str
    age:int
    roll_no:int
    nickname:str = "set"
    marks:float

class school(BaseModel):
        students:List[student]    


In [23]:
hvm=school( students=[ student(name="John Doe", age=30, roll_no=1, nickname="Johnny", marks=85.5) , student(name="Jane Smith", age=25, roll_no=2, nickname="Jenny", marks=90.0) ] )

In [25]:
hvm.students[0]

student(name='John Doe', age=30, roll_no=1, nickname='Johnny', marks=85.5)

In [26]:
class Product(BaseModel):
    name:str
    price:float
    rating:float

In [51]:
product = Product(
    name="iPhone",
    price=799,
    rating=4.5
)

__main__.Product

In [28]:
print(product)

name='iPhone' price=799.0 rating=4.5


In [ ]:
type(product.model_dump())

dict

In [29]:
print(product.model_dump())

{'name': 'iPhone', 'price': 799.0, 'rating': 4.5}


In [39]:
import json

type(json.dumps(["abc","ss"]))

str

In [46]:
data = {
    "name": "iPhone",
    "price": 799,
    "rating": 4.5
}

In [47]:
Product.model_validate(data)

Product(name='iPhone', price=799.0, rating=4.5)

In [55]:
json_data = '''
{
    "name": "iPhone",
    "price": 799,
    "rating": 4.5
}
'''
obj=Product.model_validate(json.loads(json_data))
obj

Product(name='iPhone', price=799.0, rating=4.5)

In [56]:
json_data = '''
{
    "name": "iPhone",
    "price": 799,
    "rating": 4.5
}
'''

In [57]:
product = Product.model_validate_json(json_data)

In [60]:
type(product)

__main__.Product

In [ ]:
class Product(BaseModel):
    name:str
    price:float | None = None
    rating:float | None = None
    specifications:str
    url:str|None = None
    source:str |None = None

class ProductDiscoveryResult(BaseModel):   
    products:List[Product] 

In [ ]:
json_data = '''{
    "products": [
        {
            "name": "iPhone 17",
            "price": 799,
            "rating": 4.5,
            "specifications": "256GB",
            "url": "https://apple.com",
            "source": "Apple"
        },
        {
            "name": "Pixel 10",
            "price": 699,
            "rating": 4.4,
            "specifications": "128GB",
            "url": "https://google.com",
            "source": "Google"
        }
    ]
}'''

In [70]:
dict_=json.loads(json_data)

In [72]:
print(ProductDiscoveryResult.model_validate(dict_))

products=[Product(name='iPhone 17', price=799.0, rating=4.5, specifications='256GB', url='https://apple.com', source='Apple'), Product(name='Pixel 10', price=699.0, rating=4.4, specifications='128GB', url='https://google.com', source='Google')]


# YOUTUBE LEARNINGS

created the youtube client object to interact with Yuotube via A.P.I.

In [1]:
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os

load_dotenv(override=True)

YOUTUBE_API_KEY =os.getenv("YOUTUBE_API_KEY")
## create a client to use youtube api

youtube=build(
    "youtube",
    "v3",
    developerKey=YOUTUBE_API_KEY
)


### so that later i can use.
`youtube.search()`
`youtube.videos()`
`youtube.commentThreads()`

## Search a list of videos

In [26]:
def search_videos(query,max_no_results=20):

    request=youtube.search().list(
        q=query,
        part="snippet",
        type="video",
        maxResults=max_no_results,
        order="relevance"
    )

    response=request.execute()
    videos = []
    for item in response["items"]:
        video_meta_data={
        "video_id":item["id"]["videoId"],
        "title":item["snippet"]["title"],
        "thumbnail_url":item["snippet"]["thumbnails"]["default"]["url"],
        "published": item["snippet"]["publishedAt"],
        "description": item["snippet"]["description"]
        }
        videos.append(video_meta_data)


    return videos



In [31]:
search_videos("gaming lapotop review")

[{'video_id': '_rLSZUiWsdY',
  'title': 'I EXPLORED Gaming Laptops From AMAZON in 2026',
  'thumbnail_url': 'https://i.ytimg.com/vi/_rLSZUiWsdY/default.jpg',
  'published': '2026-08-12T04:30:34Z',
  'description': 'Looking to upgrade your setup this year without breaking the bank? In this comprehensive buying guide, we break down exactly ...'},
 {'video_id': 'QnXmxFGXUCA',
  'title': 'Is This Laptop Really Worth It - Alienware 15 Gaming Laptop Review',
  'thumbnail_url': 'https://i.ytimg.com/vi/QnXmxFGXUCA/default.jpg',
  'published': '2026-08-18T16:54:32Z',
  'description': 'Alienware finally has a more accessible gaming laptop, but is the new Alienware 15 actually worth buying in 2026? In this video ...'},
 {'video_id': 'c-dgTi_g9eQ',
  'title': 'The Best Gaming Laptops for Students (2026 Edition)',
  'thumbnail_url': 'https://i.ytimg.com/vi/c-dgTi_g9eQ/default.jpg',
  'published': '2026-08-16T13:00:27Z',
  'description': 'Save BIG on Your Next Laptop: https://www.bestlaptop.deals Be

## get video stats, meta_data

In [16]:
def get_video_stats(video_id):
    request=youtube.videos().list(
        part="statistics",
        id=video_id
    )
    response=request.execute()
    stats = response["items"][0]["statistics"]
    video_stats={
        "views": int(stats.get("viewCount", 0)),
        "likes": int(stats.get("likeCount", 0)),
        "comment_count": int(stats.get("commentCount", 0))
    }
    return video_stats


In [17]:
get_video_stats("A4ctOfnNytw")

{'views': 635278, 'likes': 6260, 'comment_count': 70}

In [18]:
def get_top_comments(video_id,max_results=15):
    requests = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        order="relevance",
        maxResults=max_results,
        textFormat="plainText"
    )
    response=requests.execute()
    comments = []
    for item in response["items"]:
        top=item["snippet"]["topLevelComment"]["snippet"]
        comments.append({
        "text": top["textDisplay"],
        "likes": top["likeCount"],
        "author": top["authorDisplayName"]
        })

    return comments    

In [19]:
get_top_comments("A4ctOfnNytw")

[{'text': "They can't add gore but they can add massacre's and mass 3rd degree murder's",
  'likes': 75,
  'author': '@Hiwana_Doragon'},
 {'text': "😔Son you can't add gore to when that episode was created till the thousand year blood war they can add gore",
  'likes': 57,
  'author': '@Blastful-goat'},
 {'text': 'Son Goku here join the council or else..🫵',
  'likes': 3,
  'author': '@Songoku_council'},
 {'text': 'POV :ONE PECE ,WHITE BARD .ACE', 'likes': 4, 'author': '@TUGEGER'},
 {'text': 'I’m immune to the council. If I join then the council is supreme',
  'likes': 0,
  'author': '@Null-y8q'},
 {'text': 'Join the council', 'likes': 4, 'author': '@plueyfr0mdeltarune-s3n'},
 {'text': 'I’m immune to the council', 'likes': 7, 'author': '@Null-y8q'},
 {'text': 'also piccolo in the Android saga',
  'likes': 1,
  'author': '@kira_0398'},
 {'text': 'Who else didnr look at bottom by accident snd thought its toji',
  'likes': 0,
  'author': '@Hopefully-famous'},
 {'text': 'Toji needs this', 'l

In [24]:
from youtube_transcript_api import YouTubeTranscriptApi

yt_transcript_obj=YouTubeTranscriptApi()

def get_transcript(video_id):
    try:
        fetched_transcription=yt_transcript_obj.fetch(video_id)
        return " ".join(snippet.text for snippet in fetched_transcription)
    except Exception as e:
        print(f"Transcription error for{video_id}:{type(e).__name__}:{e}")
        return None




In [32]:
get_transcript("1ZAjYePfzzE")

"The Katana 15 is MSI's most sold gaming laptop in a lot of places. And with an Intel i9 and high power limits at a low price, this one does look impressive. But what sacrifices does it make to be cheaper? Let's find out. And please like and subscribe if you like these type of videos. From the outside, the Katana is definitely a gaming laptop. It has this black body with some pretty cool-looking accents. And while some people might not like a flashy dragon on the back of their laptop, I myself quite like this raised design. And I don't think it's too flashy. It is fully made out of plastic, though. It doesn't feel flimsy or anything, but mostly just because it's beefy cooling system is holding it all together. But beefy cooling means it weighs quite a bit. And like most gaming laptops, it's not the thinnest, either. So, I wouldn't one-hand this thing. But that cooling definitely serves a purpose. This Katana has a high power Intel Core i9 and an RTX 5060, which should be plenty for gam